# TP7 — Dashboard SOC MongoDB de supervision des accès
Dashboard professionnel pour analystes SOC : connexion MongoDB, filtrage, KPI, graphiques interactifs Plotly et export HTML autonome.


In [ ]:
import os
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from dotenv import load_dotenv
from pymongo import MongoClient
from plotly.subplots import make_subplots
from spark_jobs.risk_scoring import score_dataframe_pandas, top_risky_users
load_dotenv(ROOT / ".env")
px.defaults.template = "plotly_white"


In [ ]:
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017/")
DB_NAME = os.getenv("DB_NAME", "hopital_db")
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
db = client[DB_NAME]
client.admin.command("ping")
print(f"MongoDB connecté : {DB_NAME}")


In [ ]:
def read_collection(name):
    """Lit une collection MongoDB sans le champ technique _id."""
    return pd.DataFrame(list(db[name].find({}, {"_id": 0})))

alerts = read_collection("alerts")
if not alerts.empty:
    dashboard_df = alerts.copy()
    source = "MongoDB.alerts"
else:
    access_logs = read_collection("access_logs")
    source = "MongoDB.access_logs"
    if access_logs.empty:
        access_logs = pd.read_csv(ROOT / "datasets" / "access_logs.csv")
        source = "datasets/access_logs.csv (secours local)"
    access_logs["timestamp"] = pd.to_datetime(access_logs["timestamp"], errors="coerce")
    access_logs["success"] = access_logs["success"].astype(str).str.lower().isin(["true", "1", "yes", "oui"])
    access_logs["mfa_passed"] = access_logs["mfa_passed"].astype(str).str.lower().isin(["true", "1", "yes", "oui"])
    if "bytes" in access_logs.columns:
        access_logs["bytes"] = pd.to_numeric(access_logs["bytes"], errors="coerce").fillna(0).astype(int)
    access_logs["hour"] = access_logs["timestamp"].dt.hour
    dashboard_df = score_dataframe_pandas(access_logs)
    dashboard_df = dashboard_df[dashboard_df["risk_score"] >= 6].copy()
print(f"Source : {source} — {len(dashboard_df)} alertes élevées")


In [ ]:
# Minimisation : aucune donnée patient sensible n’est chargée ni affichée.
allowed_columns = [
    "timestamp", "user_id", "role", "department", "resource_id", "resource_type",
    "sensitivity", "action", "ip", "success", "mfa_passed", "risk_score",
    "risk_level", "risk_reasons", "alert_id",
]
dashboard_df = dashboard_df[[column for column in allowed_columns if column in dashboard_df.columns]].copy()
dashboard_df["timestamp"] = pd.to_datetime(dashboard_df["timestamp"], errors="coerce")
dashboard_df["hour"] = dashboard_df["timestamp"].dt.hour.fillna(-1).astype(int)
dashboard_df["date"] = dashboard_df["timestamp"].dt.date
dashboard_df["risk_score"] = pd.to_numeric(dashboard_df["risk_score"], errors="coerce").fillna(0).astype(int)
dashboard_df["risk_reasons_text"] = dashboard_df["risk_reasons"].apply(lambda value: " | ".join(value) if isinstance(value, list) else str(value))
dashboard_df.head()


## Paramètres interactifs SOC
Modifie les variables ci-dessous pour filtrer immédiatement tout le dashboard sans toucher aux données MongoDB. Utilise `Tous` pour conserver toutes les valeurs.


In [ ]:
SOC_USER = "Tous"          # Exemple : "u006"
SOC_ACTION = "Tous"        # Exemple : "export"
SOC_DEPARTMENT = "Tous"    # Exemple : "urgence"
SOC_MIN_SCORE = 6

filtered_df = dashboard_df.copy()
if SOC_USER != "Tous":
    filtered_df = filtered_df[filtered_df["user_id"] == SOC_USER]
if SOC_ACTION != "Tous":
    filtered_df = filtered_df[filtered_df["action"] == SOC_ACTION]
if SOC_DEPARTMENT != "Tous" and "department" in filtered_df.columns:
    filtered_df = filtered_df[filtered_df["department"] == SOC_DEPARTMENT]
filtered_df = filtered_df[filtered_df["risk_score"] >= SOC_MIN_SCORE]
print(f"Vue SOC filtrée : {len(filtered_df)} alertes")


In [ ]:
total_alerts = len(filtered_df)
users_impacted = filtered_df["user_id"].nunique() if not filtered_df.empty else 0
exports_count = int((filtered_df["action"] == "export").sum()) if not filtered_df.empty else 0
max_score = int(filtered_df["risk_score"].max()) if not filtered_df.empty else 0
kpi_html = f"""
<style>
.soc-grid {{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:10px 0 20px 0;}}
.soc-card {{background:linear-gradient(135deg,#0f172a,#1e293b);color:white;border-radius:14px;padding:18px;box-shadow:0 8px 20px rgba(15,23,42,.18);}}
.soc-label {{font-size:13px;color:#cbd5e1;text-transform:uppercase;letter-spacing:.08em;}}
.soc-value {{font-size:32px;font-weight:800;margin-top:8px;}}
.soc-note {{font-size:12px;color:#94a3b8;margin-top:6px;}}
</style>
<div class="soc-grid">
  <div class="soc-card"><div class="soc-label">Alertes</div><div class="soc-value">{total_alerts}</div><div class="soc-note">seuil ≥ {SOC_MIN_SCORE}</div></div>
  <div class="soc-card"><div class="soc-label">Utilisateurs</div><div class="soc-value">{users_impacted}</div><div class="soc-note">comptes impactés</div></div>
  <div class="soc-card"><div class="soc-label">Exports</div><div class="soc-value">{exports_count}</div><div class="soc-note">risque exfiltration</div></div>
  <div class="soc-card"><div class="soc-label">Score max</div><div class="soc-value">{max_score}</div><div class="soc-note">événement critique</div></div>
</div>
"""
display(HTML(kpi_html))


In [ ]:
top10 = pd.DataFrame(top_risky_users(filtered_df.to_dict(orient="records"), 10))
if top10.empty:
    top10 = pd.DataFrame(columns=["user_id", "total_score", "nb_alertes", "dernieres_raisons"])
top10


In [ ]:
if filtered_df.empty:
    display(HTML("<b>Aucune alerte ne correspond aux filtres SOC.</b>"))
else:
    hourly = filtered_df.groupby("hour", as_index=False).size().rename(columns={"size": "alertes"})
    by_action = filtered_df.groupby("action", as_index=False).size().rename(columns={"size": "alertes"})
    by_user = top10.sort_values("total_score", ascending=True)

    fig = make_subplots(
        rows=2, cols=2,
        specs=[[{"type": "bar"}, {"type": "pie"}], [{"type": "bar"}, {"type": "scatter"}]],
        subplot_titles=("Alertes par heure", "Répartition par action", "Top utilisateurs par score", "Chronologie des scores")
    )
    fig.add_trace(go.Bar(x=hourly["hour"], y=hourly["alertes"], marker_color="#dc2626", name="Alertes/heure", text=hourly["alertes"], textposition="outside"), row=1, col=1)
    fig.add_trace(go.Pie(labels=by_action["action"], values=by_action["alertes"], hole=.45, name="Actions"), row=1, col=2)
    fig.add_trace(go.Bar(x=by_user["total_score"], y=by_user["user_id"], orientation="h", marker_color="#2563eb", name="Score utilisateur", text=by_user["nb_alertes"], texttemplate="%{text} alertes"), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=filtered_df["timestamp"], y=filtered_df["risk_score"], mode="markers",
        marker={"size": 10, "color": filtered_df["risk_score"], "colorscale": "Reds", "showscale": True},
        customdata=filtered_df[["user_id", "action", "ip", "risk_reasons_text"]],
        hovertemplate="<b>%{customdata[0]}</b><br>Action=%{customdata[1]}<br>IP=%{customdata[2]}<br>Score=%{y}<br>%{customdata[3]}<extra></extra>",
        name="Scores"
    ), row=2, col=2)
    fig.update_layout(height=850, title_text="Dashboard SOC TP7 — Alertes d’accès à risque", title_x=.5, legend_orientation="h", hovermode="closest")
    fig.update_xaxes(title_text="Heure", row=1, col=1)
    fig.update_yaxes(title_text="Nombre d’alertes", row=1, col=1)
    fig.update_xaxes(title_text="Score cumulé", row=2, col=1)
    fig.update_yaxes(title_text="Utilisateur", row=2, col=1)
    fig.update_yaxes(title_text="Score événement", row=2, col=2)
    fig.show()


In [ ]:
if not filtered_df.empty:
    heatmap_df = filtered_df.pivot_table(index="action", columns="hour", values="risk_score", aggfunc="count", fill_value=0)
    heatmap = px.imshow(
        heatmap_df,
        labels={"x": "Heure", "y": "Action", "color": "Alertes"},
        color_continuous_scale="Reds",
        text_auto=True,
        title="Matrice SOC action × heure"
    )
    heatmap.update_layout(height=420, title_x=.5)
    heatmap.show()


In [ ]:
if filtered_df.empty:
    interpretation = "Aucune alerte élevée disponible pour les filtres SOC actuels."
else:
    leader = top10.iloc[0]
    main_action = filtered_df["action"].value_counts().idxmax()
    peak_hour = int(filtered_df["hour"].value_counts().idxmax())
    interpretation = (
        f"Priorité SOC : {leader['user_id']} cumule {leader['nb_alertes']} alertes "
        f"et un score total de {leader['total_score']}. "
        f"L’action dominante est {main_action}, avec un pic à {peak_hour}h."
    )
display(HTML(f"<div style='border-left:5px solid #dc2626;padding:12px;background:#fff7ed;font-size:15px'><b>Interprétation SOC :</b> {interpretation}</div>"))


In [ ]:
# Export optionnel pour partager le dashboard aux analystes SOC sans exposer les données patients sensibles.
EXPORT_HTML = False
if EXPORT_HTML and not filtered_df.empty:
    output_path = ROOT / "notebooks" / "dashboard_soc_tp7.html"
    fig.write_html(output_path, include_plotlyjs="cdn", full_html=True)
    print(f"Dashboard SOC exporté : {output_path}")


## Configuration de déploiement
- Renseigner `MONGO_URI` et `DB_NAME` dans l’environnement de production.
- Alimenter `access_logs` et `alerts` dans la même base MongoDB que le site.
- Garder `alerts` comme source prioritaire du dashboard notebook.
- Ne pas charger `patients_sensibles.csv` dans le dashboard.
- Pour partager un graphique aux analystes SOC, passer `EXPORT_HTML = True` dans la dernière cellule : un fichier HTML interactif sera généré localement.
